In [ ]:
# ...código anterior sin cambios...
        for idx_met, met in enumerate(mets_seleccionados):
            # ...código del ciclo for sin cambios...
            fg_met.add_to(mapa)
        nombre_mapa = os.path.join(output_dir, f'mapa_clusters_geograficos_corregido_{fecha_hora}.html')
        if mapa is not None:
            mapa.save(nombre_mapa)
            print(f'\nOK: Mapa exportado: {nombre_mapa}')
        else:
            print("ERROR: No se pudo crear el mapa porque no hay datos suficientes para los METs seleccionados.")
        if export_rows:
            print(f"\nExportando datos a Excel...")
            df_export = pd.DataFrame(export_rows)
            df_resumen_clusters = pd.DataFrame(resumen_clusters)
            resumen_general = pd.DataFrame([{'Total_METs': len(mets_seleccionados), 'Total_clusters': total_clusters_global, 'Total_clientes': total_clientes_global, 'Promedio_clientes_por_cluster': total_clientes_global / total_clusters_global if total_clusters_global > 0 else 0, 'Min_clientes_por_cluster': min([len(df_export[df_export['MET']==met]['Cluster']) for met in mets_seleccionados]) if mets_seleccionados else 0, 'Max_clientes_por_cluster': max([len(df_export[df_export['MET']==met]['Cluster']) for met in mets_seleccionados]) if mets_seleccionados else 0, 'Version': 'CORREGIDA - Todos los clusters >= 20 clientes', 'Fecha_creacion': datetime.now().strftime('%Y-%m-%d %H:%M:%S')}])
            excel_path = os.path.join(output_dir, f'clusters_geograficos_corregido_{fecha_hora}.xlsx')
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                df_export.to_excel(writer, sheet_name='Detalle_Clusters', index=False)
                df_resumen_clusters.to_excel(writer, sheet_name='Resumen_Clusters', index=False)
                resumen_general.to_excel(writer, sheet_name='Resumen_General', index=False)
            wb = openpyxl.load_workbook(excel_path)
            header_font = Font(bold=True, color='FFFFFF', size=12)
            header_fill = PatternFill('solid', fgColor='4CAF50')
            cell_fill = PatternFill('solid', fgColor='E8F5E8')
            border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
            align = Alignment(horizontal='center', vertical='center', wrap_text=True)
            for sheet_name in wb.sheetnames:
                ws = wb[sheet_name]
                for cell in ws[1]:
                    cell.font = header_font
                    cell.fill = header_fill
                    cell.alignment = align
                    cell.border = border
                for row in ws.iter_rows(min_row=2):
                    for cell in row:
                        cell.fill = cell_fill
                        cell.alignment = align
                        cell.border = border
                for col in ws.columns:
                    max_length = 0
                    col_letter = get_column_letter(col[0].column)
                    for cell in col:
                        try:
                            if cell.value:
                                max_length = max(max_length, len(str(cell.value)))
                        except:
                            pass
                    ws.column_dimensions[col_letter].width = max(15, min(max_length + 3, 45))
            wb.save(excel_path)
            print(f'OK: Excel exportado: {excel_path}')
        else:
            print("No hay datos para exportar.")
        if mapa is not None:
            display(mapa)
        print("\nProceso completado.")
# ...código posterior sin cambios...


SISTEMA LISTO - VERSION CORREGIDA
INSTRUCCIONES:
   1. Selecciona uno o varios METs
   2. Ajusta el rango de clientes por ruta (20-40)
   3. Ajusta la cantidad de clientes a analizar
   4. Presiona 'CREAR CLUSTERS Y EXPORTAR' para generar rutas, mapa y Excel


In [ ]:
# VISUALIZACION Y EXPORTACION DE CLUSTERS
# Genera mapa interactivo y exporta resultados a Excel
# EJECUTA PRIMERO LA CELDA ANTERIOR PARA CREAR LOS CLUSTERS

import openpyxl
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
import os
from datetime import datetime
import folium
import random

# Configuracion de directorios
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\Output'
os.makedirs(output_dir, exist_ok=True)
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Ceylan\95_24.png'
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

# Verificar que existen METs seleccionados y clusters creados
try:
    mets_seleccionados = [met.description for met in met_checkboxes if met.value]
except:
    print("ERROR: No se encontraron METs seleccionados. Ejecuta primero la celda anterior.")
    mets_seleccionados = []

if not mets_seleccionados:
    print("ERROR: No hay METs seleccionados. Ejecuta primero la celda anterior para crear clusters.")
else:
    print(f"Creando visualizacion para {len(mets_seleccionados)} METs...")
    
    # Verificar que existen los clusters
    clusters_encontrados = []
    for met in mets_seleccionados:
        if f'clusters_{met}' in globals() and f'poligonos_{met}' in globals():
            clusters_encontrados.append(met)
    
    if not clusters_encontrados:
        print("ERROR: No se encontraron clusters creados. Ejecuta 'CREAR CLUSTERS' en la celda anterior.")
    else:
        print(f"OK: Clusters encontrados para: {clusters_encontrados}")
        
        # CREAR MAPA INTERACTIVO
        # Inicializar mapa centrado en el primer MET
        if not df_cdr.empty and clusters_encontrados:
            met_row = df_cdr[df_cdr['CodMET'] == clusters_encontrados[0]].iloc[0]
            mapa = folium.Map(
                location=[met_row['U_latitud'], met_row['U_longitud']], 
                zoom_start=10, 
                tiles='OpenStreetMap'
            )
        else:
            mapa = folium.Map(location=[20.5, -100.8], zoom_start=7, tiles='OpenStreetMap')

        # Paleta de colores para clusters
        colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 
                   'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 
                   'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

        # Variables para estadisticas y exportacion
        export_rows = []
        resumen_clusters = []
        total_clientes_global = 0
        total_clusters_global = 0
        featuregroups_met = {}

        def normalizar_id(valor):
            return str(valor).replace(' ', '').replace(':', '').replace('.', '').replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u').replace('ñ','n')

        # PROCESAR CADA MET
        for idx_met, met in enumerate(clusters_encontrados):
            print(f"\nVisualizando MET {idx_met+1}/{len(clusters_encontrados)}: {met}")
            
            clusters = globals()[f'clusters_{met}']
            poligonos = globals()[f'poligonos_{met}']
            
            print(f"   OK: {len(clusters)} clusters con {len(poligonos)} poligonos")
            
            # Obtener informacion del MET
            if not df_cdr.empty:
                met_row = df_cdr[df_cdr['CodMET'] == met].iloc[0]
            else:
                continue
            
            # Crear FeatureGroup para este MET
            met_id = normalizar_id(met)
            fg_met = folium.FeatureGroup(
                name=f"MET {met} ({len(clusters)} clusters)", 
                show=True if idx_met==0 else False
            )
            featuregroups_met[met] = fg_met
            
            # Estadisticas por MET
            total_clientes_met = sum(len(cluster) for cluster in clusters)
            total_clusters_global += len(clusters)
            total_clientes_global += total_clientes_met
            
            # DIBUJAR CLUSTERS EN EL MAPA
            for idx_cluster, (cluster, poligono_info) in enumerate(zip(clusters, poligonos)):
                color_cluster = colores[idx_cluster % len(colores)]
                
                # Dibujar poligono del cluster
                if hasattr(poligono_info['poligono'], 'exterior'):
                    # Poligono valido
                    coords_poligono = list(poligono_info['poligono'].exterior.coords)
                    coords_folium = [(lat, lon) for lon, lat in coords_poligono]
                    
                    folium.Polygon(
                        locations=coords_folium,
                        color=color_cluster,
                        weight=3,
                        fillColor=color_cluster,
                        fillOpacity=0.2,
                        popup=f"MET {met} - Cluster {idx_cluster+1}<br>Clientes: {len(cluster)}<br>Centroide: {poligono_info['centroid']['U_latitud']:.4f}, {poligono_info['centroid']['U_longitud']:.4f}"
                    ).add_to(fg_met)
                else:
                    # Buffer (circulo)
                    centroid = poligono_info['centroid']
                    folium.Circle(
                        location=[centroid['U_latitud'], centroid['U_longitud']],
                        radius=1000,
                        color=color_cluster,
                        weight=3,
                        fillColor=color_cluster,
                        fillOpacity=0.2,
                        popup=f"MET {met} - Cluster {idx_cluster+1}<br>Clientes: {len(cluster)}"
                    ).add_to(fg_met)
                
                # Marcadores de clientes en el cluster
                for idx_cliente, (_, cliente) in enumerate(cluster.iterrows()):
                    popup_html = f'''
                    <div style="background: #fff; border-radius: 16px; box-shadow: 0 2px 8px rgba(0,0,0,0.15); padding: 14px; border: 2px solid {color_cluster}; min-width: 260px;">
                        <div style="font-size:18px; color:{color_cluster}; font-weight:bold; margin-bottom:6px;">
                            Cliente: <span style="color:{color_cluster};">{cliente.get('CodCli', 'N/A')}</span>
                        </div>
                        <div style="font-size:14px; color:#333; margin-bottom:4px;">
                            <b>Nombre:</b> {cliente.get('Nombre', 'N/A')}
                        </div>
                        <div style="font-size:14px; color:#2A81CB; margin-bottom:4px;">
                            <b>MET:</b> {met}
                        </div>
                        <div style="font-size:14px; color:#4CAF50; margin-bottom:4px;">
                            <b>Cluster:</b> {idx_cluster+1} de {len(clusters)}
                        </div>
                        <div style="font-size:13px; color:#666; margin-bottom:4px;">
                            <b>Coords:</b> {cliente['U_latitud']:.4f}, {cliente['U_longitud']:.4f}
                        </div>
                        <div style="font-size:13px; color:#FF9800;">
                            <b>Clientes en cluster:</b> {len(cluster)}
                        </div>
                    </div>
                    '''
                    
                    folium.Marker(
                        location=[cliente['U_latitud'], cliente['U_longitud']],
                        popup=folium.Popup(popup_html, max_width=340),
                        icon=folium.Icon(color=color_cluster, icon='user')
                    ).add_to(fg_met)
                    
                    # Numeracion del cliente en el cluster
                    folium.Marker(
                        location=[cliente['U_latitud'], cliente['U_longitud']],
                        icon=DivIcon(
                            icon_size=(20,20),
                            icon_anchor=(10,10),
                            html=f'<div style="font-size:12px; color:white; background:{color_cluster}; border-radius:50%; width:20px; height:20px; text-align:center; line-height:20px; border:1px solid #fff; font-weight:bold;">{idx_cliente+1}</div>'
                        )
                    ).add_to(fg_met)
                
                # Centroide del cluster
                centroid = poligono_info['centroid']
                folium.Marker(
                    location=[centroid['U_latitud'], centroid['U_longitud']],
                    popup=f"Centroide Cluster {idx_cluster+1}<br>MET: {met}<br>Clientes: {len(cluster)}",
                    icon=folium.Icon(color=color_cluster, icon='star', prefix='fa')
                ).add_to(fg_met)
                
                # PREPARAR DATOS PARA EXCEL
                for idx_cliente, (_, cliente) in enumerate(cluster.iterrows()):
                    export_rows.append({
                        'MET': met,
                        'Cluster': f"Cluster_{idx_cluster+1}",
                        'Numero_en_cluster': idx_cliente+1,
                        'Codigo_cliente': cliente.get('CodCli', 'N/A'),
                        'Nombre_cliente': cliente.get('Nombre', 'N/A'),
                        'Latitud': cliente['U_latitud'],
                        'Longitud': cliente['U_longitud'],
                        'Total_clientes_cluster': len(cluster),
                        'Centroide_lat': centroid['U_latitud'],
                        'Centroide_lon': centroid['U_longitud']
                    })
                
                # Resumen por cluster
                resumen_clusters.append({
                    'MET': met,
                    'Cluster': f"Cluster_{idx_cluster+1}",
                    'Num_clientes': len(cluster),
                    'Centroide_lat': centroid['U_latitud'],
                    'Centroide_lon': centroid['U_longitud'],
                    'Color_visualizacion': color_cluster
                })
            
            # Marcador del MET
            if os.path.exists(icon_met_path):
                folium.Marker(
                    location=[met_row['U_latitud'], met_row['U_longitud']],
                    popup=folium.Popup(f'''
                    <div style="background:#2A81CB; color:#fff; border-radius:14px; box-shadow:0 2px 8px rgba(0,0,0,0.15); padding:14px; border:2px solid #fff; min-width:220px; text-align:center;">
                        <div style="font-size:18px; font-weight:bold; margin-bottom:6px;">MET: <span style="color:#FFD600;">{met}</span></div>
                        <div style="font-size:14px; color:#fff; margin-bottom:4px;">Total clusters: {len(clusters)}</div>
                        <div style="font-size:14px; color:#fff; margin-bottom:4px;">Total clientes: {total_clientes_met}</div>
                        <div style="font-size:12px; color:#FFD600;">{met_row['U_latitud']:.4f}, {met_row['U_longitud']:.4f}</div>
                    </div>
                    ''', max_width=260),
                    icon=CustomIcon(icon_met_path, icon_size=(40,40))
                ).add_to(fg_met)
            else:
                folium.Marker(
                    location=[met_row['U_latitud'], met_row['U_longitud']],
                    popup=f"MET {met}<br>Clusters: {len(clusters)}<br>Clientes: {total_clientes_met}",
                    icon=folium.Icon(color='red', icon='home', prefix='fa')
                ).add_to(fg_met)

        # FINALIZAR MAPA
        # Agregar todas las capas al mapa
        for fg in featuregroups_met.values():
            fg.add_to(mapa)
        
        # Control de capas
        folium.LayerControl(collapsed=False, autoZIndex=True).add_to(mapa)

        # Titulo del mapa
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background-color: white; padding: 10px 25px; border: 2px solid #333; border-radius: 15px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);">
            <h2 style="margin: 0; color: #333; text-align: center; font-size:16px;">
                MAPA DE CLUSTERS GEOGRAFICOS (VERSION CORREGIDA)
            </h2>
            <p style="margin: 6px 0 0 0; text-align: center; color: #666; font-size: 11px;">
                METs: <b>{len(clusters_encontrados)}</b> | Clusters: <b>{total_clusters_global}</b> | Clientes: <b>{total_clientes_global}</b>
            </p>
        </div>
        '''
        mapa.get_root().html.add_child(folium.Element(titulo_html))

        # Panel de estadisticas
        estadisticas_html = f'''
        <div style="position: fixed; top: 20px; right: 20px; width: 350px; background-color: white; border: 2px solid #333; z-index: 9997; font-size: 12px; padding: 15px; border-radius: 15px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);">
            <h3 style="margin-top: 0; color: #2A81CB; text-align: center; font-size:16px;">ESTADISTICAS - VERSION CORREGIDA</h3>
            <div style="background: #f8f9fa; padding: 10px; border-radius: 8px; margin-bottom: 10px;">
                <p style="margin: 4px 0; font-weight: bold; color: #333;">Total de clusters: {total_clusters_global}</p>
                <p style="margin: 4px 0; font-weight: bold; color: #333;">Total de clientes: {total_clientes_global}</p>
                <p style="margin: 4px 0; font-weight: bold; color: #333;">METs procesados: {len(clusters_encontrados)}</p>
                <p style="margin: 4px 0; font-weight: bold; color: #4CAF50;">Todos los clusters >= 20 clientes</p>
            </div>
        '''
        
        # Estadisticas por MET
        for met in clusters_encontrados:
            clusters = globals()[f'clusters_{met}']
            total_clientes_met = sum(len(cluster) for cluster in clusters)
            promedio_clientes = total_clientes_met / len(clusters) if clusters else 0
            tamaños = [len(c) for c in clusters]
            min_cluster = min(tamaños) if tamaños else 0
            max_cluster = max(tamaños) if tamaños else 0
            
            estadisticas_html += f'''
            <div style="border-left: 4px solid #4CAF50; padding-left: 10px; margin-bottom: 8px;">
                <p style="margin: 2px 0; color: #4CAF50; font-weight: bold;">MET {met}</p>
                <p style="margin: 2px 0; color: #666; font-size: 11px;">  {len(clusters)} clusters | {total_clientes_met} clientes</p>
                <p style="margin: 2px 0; color: #666; font-size: 11px;">  Rango: {min_cluster}-{max_cluster} clientes</p>
                <p style="margin: 2px 0; color: #666; font-size: 11px;">  Promedio: {promedio_clientes:.1f} clientes/cluster</p>
            </div>
            '''
        
        estadisticas_html += '</div>'
        mapa.get_root().html.add_child(folium.Element(estadisticas_html))

        # GUARDAR MAPA
        nombre_mapa = os.path.join(output_dir, f'mapa_clusters_geograficos_corregido_{fecha_hora}.html')
        mapa.save(nombre_mapa)
        print(f'\nOK: Mapa exportado: {nombre_mapa}')

        # EXPORTAR A EXCEL
        if export_rows:
            print(f"\nExportando datos a Excel...")
            
            # DataFrames para Excel
            df_export = pd.DataFrame(export_rows)
            df_resumen_clusters = pd.DataFrame(resumen_clusters)
            
            # Resumen general
            resumen_general = pd.DataFrame([{
                'Total_METs': len(clusters_encontrados),
                'Total_clusters': total_clusters_global,
                'Total_clientes': total_clientes_global,
                'Promedio_clientes_por_cluster': total_clientes_global / total_clusters_global if total_clusters_global > 0 else 0,
                'Min_clientes_por_cluster': min([len(globals()[f'clusters_{met}'][i]) for met in clusters_encontrados for i in range(len(globals()[f'clusters_{met}']))]) if clusters_encontrados else 0,
                'Max_clientes_por_cluster': max([len(globals()[f'clusters_{met}'][i]) for met in clusters_encontrados for i in range(len(globals()[f'clusters_{met}']))]) if clusters_encontrados else 0,
                'Version': 'CORREGIDA - Todos los clusters >= 20 clientes',
                'Fecha_creacion': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }])
            
            # Guardar Excel
            excel_path = os.path.join(output_dir, f'clusters_geograficos_corregido_{fecha_hora}.xlsx')
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                df_export.to_excel(writer, sheet_name='Detalle_Clusters', index=False)
                df_resumen_clusters.to_excel(writer, sheet_name='Resumen_Clusters', index=False)
                resumen_general.to_excel(writer, sheet_name='Resumen_General', index=False)

            # Formatear Excel
            wb = openpyxl.load_workbook(excel_path)
            header_font = Font(bold=True, color='FFFFFF', size=12)
            header_fill = PatternFill('solid', fgColor='4CAF50')
            cell_fill = PatternFill('solid', fgColor='E8F5E8')
            border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
            align = Alignment(horizontal='center', vertical='center', wrap_text=True)

            for sheet_name in wb.sheetnames:
                ws = wb[sheet_name]
                
                # Formatear encabezados
                for cell in ws[1]:
                    cell.font = header_font
                    cell.fill = header_fill
                    cell.alignment = align
                    cell.border = border

                # Formatear celdas de datos
                for row in ws.iter_rows(min_row=2):
                    for cell in row:
                        cell.fill = cell_fill
                        cell.alignment = align
                        cell.border = border

                # Ajustar ancho de columnas
                for col in ws.columns:
                    max_length = 0
                    col_letter = get_column_letter(col[0].column)
                    for cell in col:
                        try:
                            if cell.value:
                                max_length = max(max_length, len(str(cell.value)))
                        except:
                            pass
                    ws.column_dimensions[col_letter].width = max(15, min(max_length + 3, 45))

            wb.save(excel_path)
            print(f'OK: Excel exportado: {excel_path}')
            
            # RESUMEN FINAL
            print(f"\n{'='*80}")
            print(f"CLUSTERING GEOGRAFICO COMPLETADO - VERSION CORREGIDA")
            print(f"{'='*80}")
            print(f"METs procesados: {len(clusters_encontrados)}")
            print(f"Clusters creados: {total_clusters_global}")
            print(f"Clientes agrupados: {total_clientes_global}")
            print(f"Promedio clientes/cluster: {total_clientes_global / total_clusters_global if total_clusters_global > 0 else 0:.1f}")
            
            # Verificar que todos los clusters cumplen minimo
            todos_los_tamaños = []
            for met in clusters_encontrados:
                clusters = globals()[f'clusters_{met}']
                todos_los_tamaños.extend([len(c) for c in clusters])
            
            if todos_los_tamaños:
                min_real = min(todos_los_tamaños)
                max_real = max(todos_los_tamaños)
                print(f"Rango real: {min_real}-{max_real} clientes por cluster")
                if min_real >= 20:
                    print(f"OK: TODOS LOS CLUSTERS CUMPLEN MINIMO DE 20 CLIENTES")
                else:
                    print(f"ERROR: ALGUNOS CLUSTERS NO CUMPLEN MINIMO: {min_real} < 20")
            
            print(f"Mapa: {nombre_mapa}")
            print(f"Excel: {excel_path}")
            print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"Version: CORREGIDA - Bug de clusters pequeños solucionado")
            print(f"{'='*80}")
            
        else:
            print("No hay datos para exportar.")

        # Mostrar el mapa
        display(mapa)